# Functional analysis

- Normalize ribo signal
- Plot taxonomy plots
- Plot functional summaries
- Test KEGG category enrichment

In [18]:
import json
from pathlib import Path

import pandas as pd

from src.helpers.functional_analysis_helper import normalize_ribo, plot_taxonomy, taxonomy_read_length, plot_functional, test_kegg_categories, plot_kegg_test

Set the metagenome id and ribo id to your chosen sample.

In [19]:
RIBO_ID = "RIBO_ID"
MANIFEST_PATH = Path("file_manifest.json")
manifest = json.loads(MANIFEST_PATH.read_text())
ribo_bam = manifest["ribo"]["alignment"][RIBO_ID]
cds_counts_path = f"output/coverage/{RIBO_ID}/{RIBO_ID}_cds_counts.tsv"

master_df = pd.read_csv(cds_counts_path, sep="\t")

master_df.set_index("CDS_id", inplace=True)

Normalize ribo counts per CDS ($\frac{\mathrm{RPKM}}{\mathrm{Mean\ Metagenomic\ Coverage}}$)

In [20]:
master_df['normalized_ribo'] = normalize_ribo(master_df)
taxonomic_df = master_df[master_df['genus'].fillna('').astype(str).str.strip().ne('')].copy()

Set `OTHER_CUTOFF` to the maximum proportion placed in the 'Other' genus group; then plot taxonomy distributions

In [21]:
OTHER_CUTOFF = 0.005
meta_taxo_fig, ribo_taxo_fig = plot_taxonomy(taxonomic_df, OTHER_CUTOFF)
meta_taxo_fig.show()
ribo_taxo_fig.show()

Plot taxonomy-stratified read length distributions.

In [22]:
rdlen_fig = taxonomy_read_length(taxonomic_df, ribo_bam)
rdlen_fig.show()

Plot functional summary.

In [23]:
BRITE3 = r"/path/to/ko_to_brite_3.tsv"
TOP_N = 15
TOP_S = 10
FONTSIZE = 14

func_fig = plot_functional(master_df, BRITE3, TOP_N, TOP_S, "KEGG Categories with Highest Translational Activity", FONTSIZE)
func_fig.show()

Run two-sided binomial tests for all mapped KEGG categories.

In [24]:
kegg_results = test_kegg_categories(master_df, BRITE3)
kegg_test_fig = plot_kegg_test(kegg_results)
kegg_test_fig.show()